In [ ]:
from selenium import webdriver
from bs4 import BeautifulSoup as bs
import pandas as pd
import requests

In [ ]:
# 무신사의 반팔 상의 주소를 저장 
url = 'https://www.musinsa.com/category/001001/goods?gf=A'

# 상품에 대한 정보(브랜명, 상품 이름, 상품의 할인율, 가격)

res = requests.get(url)
res

In [ ]:
soup = bs(res.text, 'html.parser')
soup

1. selenium을 이용하여 무신사 페이지에 요청 
2. 해당 페이지의 html 문서를 불러온다. 
3. bs4를 이용하여 데이터 파싱 
4. GoodsList__List로 시작하는 class 값을 가진 div 태그를 찾는다. 
5. GoodsList__Row로 시작하는 class 값을 가진 모든 div 태그를 찾는다. 
6. sc-it로 시작하는 class 값을 가진 모든 div 태그를 찾는다. 
7. 브랜드명, 이름, 할인율, 가격 데이터, 해당 상품의 링크 주소를 추출
8. 추출한 데이터를 데이터프레임으로 생성

In [ ]:
import re

In [ ]:
driver = webdriver.Chrome()

In [ ]:
# 무신사 페이지에 요청 
driver.get(url)

In [ ]:
soup = bs(driver.page_source, 'html.parser')

In [ ]:
soup

In [ ]:
# class이 값이 특정 문자로 시작하는? 
# re.compile(r"^GoddsList__List")
div_tag = soup.find('div', class_=re.compile(r"^GoodsList__List")) 

In [ ]:
goods_row = div_tag.find_all('div', class_= re.compile(r"^GoodsList__Row"))

In [ ]:
# goods_row에서 각각의 원소에서 sc-it로 시작하는 class 값을 가진 div 태그들을 모두 찾는다. 
goods_dict = []
for good_info in goods_row:
    info_data = good_info.find_all('div', class_=re.compile(r'^sc-it'))
    for data in info_data:
        # 상품의 정보를 저장할수 있는 딕셔너리형 데이터 초기값을 저장 
        info_dict = {}
        # info_dict에서 사용할 키 값들의 목록 생성 
        dict_keys = ['브랜드', '상품명', '할인율', '판매가격']
        # print( len(data.find_all('span')) )
        span_tags = data.find_all('span')
        for span, k in zip(span_tags, dict_keys):
            text = span.get_text()
            # info_dict에 데이터를 추가 
            info_dict[k] = text.strip()
            # print(info_dict)
        # 해당 상품의 하이퍼링크 주소 값을 info_dict에 추가 
        # data에서 a태그들을 찾아서 2번째 a 태그의 href 값을 추출
        link_url = data.find_all('a')[1]['href']
        info_dict['link'] = link_url
        # break
        # 3번째 반복문이 끝나고 만들어진 상품 정보 데이터를 goods_dict에 추가 
        goods_dict.append(info_dict)
    # break

In [ ]:
goods_dict

In [ ]:
# driver에서 화면 스크롤 이벤트 
driver.execute_script(
    'window.scrollBy(0, 1600);'
)

In [ ]:
df = pd.DataFrame(goods_dict)
df

In [47]:
link_list = df['link'].tolist()
name_list = df['상품명'].tolist()
link_list[0]

'https://www.musinsa.com/products/6659876'

In [ ]:
driver = webdriver.Chrome()

In [ ]:
driver.get(link_list[0])

In [ ]:
# 스크롤을 마지막까지 내린다. 
driver.execute_script(
    'window.scrollBy(0, document.body.scrollHeight);'
)

In [ ]:
soup2 = bs(driver.page_source, 'html.parser')

In [ ]:
# GoodsReviewListSection 시작하는 class 값을 가진 div 태그를 선택 
reviews_tag = soup2.find('div', class_=re.compile(f"^GoodsReviewListSection"))
reviews_tag

In [ ]:
# ExpandableContent 시작하는 class 값을 가진 div 태그를 모두 찾는다. 
review_tags = reviews_tag.find_all('div', class_=re.compile(r"^ExpandableContent"))
review_tags

In [ ]:
[ tag.get_text().replace('\n', '') for tag in review_tags ]

In [53]:
import time

In [51]:
driver = webdriver.Chrome()


In [ ]:
# 반복문 생성 (상품명 리스트와 link를 이용해서)
review_dict =[]
for name, link in zip(name_list, link_list):
    # print(link)
    # break
    driver.get(link)
    time.sleep(1)
    soup2 = bs(driver.page_source, 'html.parser')
    reviews_tag = soup2.find('div', class_=re.compile(f"^GoodsReviewListSection"))
    try:
        review_tags = reviews_tag.find_all('div', class_=re.compile(r"^ExpandableContent"))
        for tag in review_tags:
            text = tag.get_text().replace('\n', '')
            review_dict.append(
                {
                    '상품명' : name, 
                    '리뷰' : text
                }
            )
    except:
        pass

In [55]:
driver.close()

In [57]:
review_df = pd.DataFrame(review_dict)

In [59]:
# 중복 리뷰는 제거 
review_df.drop_duplicates('리뷰', inplace=True)

In [60]:
# df 와 review_df를 결합(조인 결합)

total_df = pd.merge(df, review_df, on = '상품명', how='left') 

In [61]:
total_df

,브랜드,상품명,할인율,판매가격,link,리뷰
0,소버먼트,【건조기 가능】 3-Way 롤업 레이어드 소프트 티셔츠 [9 Color],40%,"24,900원",https://www.musinsa.com/products/6659876,피그먼트 반팔이랑 다르게 재질이 부드러운 편이지만 한여름에도 편하게 잘 입을것 같습...
1,소버먼트,【건조기 가능】 3-Way 롤업 레이어드 소프트 티셔츠 [9 Color],40%,"24,900원",https://www.musinsa.com/products/6659876,피그먼트 반팔이랑 다르게 재질이 부드러운 편이어서한여름에도 편하게 잘 입을것 같습니...
2,소버먼트,【건조기 가능】 3-Way 롤업 레이어드 소프트 티셔츠 [9 Color],40%,"24,900원",https://www.musinsa.com/products/6659876,피그먼트 반팔이랑 다르게 재질이 부드러운 편이어서 한여름에도 편하게 잘 입을것 같습...
3,누아트 스튜디오,[송승일 PICK] 우먼 솔리드 반팔 티셔츠_4colors,56%,"13,770원",https://www.musinsa.com/products/6433549,체험단 이 후기는 제품을 무상으로 제공받아 직접 사용해 본 후 작성되었습니다.여름에...
4,누아트 스튜디오,[송승일 PICK] 우먼 솔리드 반팔 티셔츠_4colors,56%,"13,770원",https://www.musinsa.com/products/6433549,체험단 이 후기는 제품을 무상으로 제공받아 직접 사용해 본 후 작성되었습니다.배송은...
...,...,...,...,...,...,...
90,카네이테이,[현킴 X KNT] PIG PARM CROP SHORT SLEEVE (PIGMENT...,21%,"46,900원",https://www.musinsa.com/products/6592529,크롭한 기장감이 옷이너무 관찮고 이뻤습니다 잘입을기여크롭한 기장감이 옷이너무 관찮고...
91,벤힛,Heart Dream 슬림핏 반팔티,49%,"24,720원",https://www.musinsa.com/products/6616076,민트 티셔츠가 가지구 싶었는데 예뻐요! 사진보다는 허리를 좀 더 잡아주는 핏이에요민...
92,벤힛,Heart Dream 슬림핏 반팔티,49%,"24,720원",https://www.musinsa.com/products/6616076,색깔이 화면이랑 똑같이 예뻐용!!여름에 시원하게 입기 좋은 색감이에요다만 사이즈가 ...
93,토아베,Statement Floral T-Shirt (2 Color),25%,"47,250원",https://www.musinsa.com/products/6621658,NaN


In [62]:
len(review_df['상품명'].unique())

21